# Lezione 31 — Self-attention, la matematica

> **Modulo:** Transformer e modello open · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 30 (attenzione come recupero morbido).
>
> Obiettivo pratico unico: costruire da zero, in NumPy, la **self-attention**
> — dove query, chiavi e valori sono *proiezioni della stessa sequenza* — e
> leggere la matrice di attenzione posizione-per-posizione.

## Parte 1 — Da attenzione a *self*-attention

Nella Lezione 30 la query veniva da *fuori* (una nuova memoria) e chiavi/valori
da una banca. Nella **self-attention** query, chiavi e valori nascono tutti
dalla **stessa** sequenza di token, tramite tre matrici di pesi imparate
$W_Q, W_K, W_V$:

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$

poi si applica la stessa formula della Lezione 30:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Il risultato: ogni token produce un nuovo vettore che e' una **media pesata di
tutti i token della sequenza, incluso se stesso**. Da qui il nome *self*. La
matrice di attenzione e' ora **quadrata** ($n \times n$, con $n$ = lunghezza
della sequenza): la riga *i* dice quanto il token *i* guarda ciascun token *j*.

In [1]:
import numpy as np

rng = np.random.default_rng(31)


def softmax(scores, axis=-1):
    scores = scores - np.max(scores, axis=axis, keepdims=True)
    exp = np.exp(scores)
    return exp / np.sum(exp, axis=axis, keepdims=True)


def self_attention(X, W_Q, W_K, W_V):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (n, n): matrice quadrata
    A = softmax(scores, axis=-1)        # ogni riga somma a 1
    out = A @ V
    return out, A

## Parte 2 — Una frase-memoria come sequenza

Prendo una memoria del progetto, la spezzo in token (parole) e do a ogni token
un vettore. Come nelle Lezioni 30, uso un embedding deterministico (hashing) —
niente modello esterno, cosi' gira nell'ambiente di base.

In [2]:

frase = "Marco visited Glasgow with his son"
tokens = frase.lower().split()
n = len(tokens)

DIM = 16


def embed_token(tok, dim=DIM):
    vec = np.zeros(dim)
    for i, ch in enumerate(tok):
        vec[(ord(ch) + i) % dim] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


X = np.vstack([embed_token(t) for t in tokens])   # (n, DIM)
print("token:", tokens)
print("forma della sequenza X:", X.shape)

token: ['marco', 'visited', 'glasgow', 'with', 'his', 'son']
forma della sequenza X: (6, 16)


In [3]:
# matrici di proiezione (qui casuali: in un vero Transformer sono imparate)
d_k = 8
W_Q = rng.normal(size=(DIM, d_k)) * 0.5
W_K = rng.normal(size=(DIM, d_k)) * 0.5
W_V = rng.normal(size=(DIM, d_k)) * 0.5

out, A = self_attention(X, W_Q, W_K, W_V)
print("forma output:", out.shape, "(un nuovo vettore per token)")
print("forma matrice di attenzione:", A.shape, "(quadrata: token x token)")
print("ogni riga somma a 1:", np.allclose(A.sum(axis=1), 1.0))

forma output: (6, 8) (un nuovo vettore per token)
forma matrice di attenzione: (6, 6) (quadrata: token x token)
ogni riga somma a 1: True


### Leggere la matrice di attenzione

La riga *i* mostra come il token *i* distribuisce la sua attenzione su tutti i
token. Stampiamola in modo leggibile.

In [4]:
np.set_printoptions(precision=2, suppress=True)
print("        " + "  ".join(f"{t[:5]:>5}" for t in tokens))
for i, t in enumerate(tokens):
    riga = "  ".join(f"{A[i, j]:5.2f}" for j in range(n))
    print(f"{t[:6]:>6}  {riga}")

# per ogni token, su chi si concentra di piu'
print()
for i, t in enumerate(tokens):
    j = int(np.argmax(A[i]))
    print(f"'{t}' guarda soprattutto '{tokens[j]}' (peso {A[i, j]:.2f})")

        marco  visit  glasg   with    his    son
 marco   0.17   0.18   0.16   0.16   0.17   0.16
visite   0.16   0.16   0.17   0.16   0.18   0.16
glasgo   0.17   0.19   0.16   0.16   0.17   0.14
  with   0.14   0.16   0.18   0.19   0.18   0.15
   his   0.17   0.17   0.16   0.15   0.19   0.16
   son   0.19   0.18   0.17   0.16   0.15   0.16

'marco' guarda soprattutto 'visited' (peso 0.18)
'visited' guarda soprattutto 'his' (peso 0.18)
'glasgow' guarda soprattutto 'visited' (peso 0.19)
'with' guarda soprattutto 'with' (peso 0.19)
'his' guarda soprattutto 'his' (peso 0.19)
'son' guarda soprattutto 'marco' (peso 0.19)


## Parte 3 — Il progetto: Memory AI Lab, passo 31

Impacchetto la self-attention in una funzione riusabile che, data una frase
memoria, restituisce la rappresentazione *contestualizzata* di ogni token
(ogni vettore ora "sa" degli altri token). E' il pezzo che la Lezione 32
inserira' dentro un blocco Transformer completo.

In [5]:
def contestualizza(frase, dim=DIM, d_k=8, seed=31):
    r = np.random.default_rng(seed)
    toks = frase.lower().split()
    X = np.vstack([embed_token(t, dim) for t in toks])
    Wq = r.normal(size=(dim, d_k)) * 0.5
    Wk = r.normal(size=(dim, d_k)) * 0.5
    Wv = r.normal(size=(dim, d_k)) * 0.5
    out, A = self_attention(X, Wq, Wk, Wv)
    return toks, out, A


toks, ctx, A = contestualizza("The user prefers morning sessions")

# controllo di non-regressione: una rappresentazione per token, attenzione valida
assert ctx.shape[0] == len(toks)
assert np.allclose(A.sum(axis=1), 1.0)
assert A.shape == (len(toks), len(toks))
print("token contestualizzati:", ctx.shape)
print("matrice di attenzione:", A.shape, "- righe normalizzate OK")

token contestualizzati: (5, 8)
matrice di attenzione: (5, 5) - righe normalizzate OK


## Riepilogo (max 8 punti)

1. **Self**-attention: Q, K, V sono proiezioni della *stessa* sequenza
   ($Q=XW_Q$, ecc.).
2. Ogni token diventa una media pesata di *tutti* i token, incluso se stesso.
3. La matrice di attenzione e' **quadrata** $n\times n$.
4. La riga *i* = come il token *i* distribuisce attenzione sugli altri.
5. Le matrici $W_Q,W_K,W_V$ sono **imparate** (qui casuali per illustrare).
6. La formula e' identica alla Lezione 30: solo la provenienza di Q/K/V cambia.
7. L'output ha una riga per token: rappresentazioni **contestualizzate**.
8. E' il cuore del blocco Transformer (Lezione 32).

## Quiz

1. Perche' la matrice di attenzione della self-attention e' quadrata, mentre
   quella della Lezione 30 no?
2. Cosa rappresenta l'elemento $A_{ij}$?
3. Se $W_Q=W_K=W_V=I$ (identita'), la self-attention sparisce?

*(Risposte: 1. perche' query e chiavi vengono dalla stessa sequenza, quindi
$n$ query e $n$ chiavi; 2. quanto il token $i$ attende al token $j$; 3. no —
resta $\text{softmax}(XX^\top/\sqrt d)X$, cioe' attenzione basata sulla
similarita' grezza tra i token.)*